## Epigenetic Data Extraction Pipeline

In [ ]:
"""
Parses raw sequencing alignments (.bam) and processed signal tracks (.bigWig) 
to extract multi-dimensional epigenetic features across genomic bins.
"""
import os
import pyBigWig
import pysam
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# Initialize directory architecture
os.makedirs("../data/inputs/bigwig", exist_ok=True)
os.makedirs("../data/inputs/bam", exist_ok=True)
os.makedirs("../data/saves", exist_ok=True)

def process_bigWig(file_path, bin_size=100_000):
    """
    Scans a BigWig file and extracts statistical signal summaries 
    (mean, max, min, std, median, coverage) for fixed-size genomic bins across autosomes.
    """
    bw = pyBigWig.open(file_path)
    
    valid_chroms = [chrom for chrom in bw.chroms() if chrom.startswith("chr") and "_" not in chrom]
    autosomes = [f"chr{i}" for i in range(1, 23)]
    if 'chrX' in valid_chroms:
        autosomes.append('chrX')
        
    chroms_to_use = [chrom for chrom in autosomes if chrom in valid_chroms]
    records = []
    
    for chrom in chroms_to_use:
        chrom_len = bw.chroms()[chrom]
        for start in range(0, chrom_len, bin_size):
            end = min(start + bin_size, chrom_len)
            values = bw.values(chrom, start, end, numpy=True)
            
            clean_vals = values[~np.isnan(values)]
            signal_mean = np.nanmean(values)
            
            records.append({
                "chrom": chrom,
                "start": start,
                "end": end,
                "signal": signal_mean if not np.isnan(signal_mean) else 0,
                "max_signal": np.nanmax(values) if len(clean_vals) > 0 else 0,
                "min_signal": np.nanmin(values) if len(clean_vals) > 0 else 0,
                "std_signal": np.nanstd(values) if len(clean_vals) > 0 else 0,
                "median_signal": np.nanmedian(values) if len(clean_vals) > 0 else 0,
                "coverage": np.count_nonzero(~np.isnan(values))
            })
            
    bw.close()
    return pd.DataFrame(records)

def get_bam_coverage_per_bin(bam_path, bins_df):
    """
    Iterates through established genomic bins to extract read coverage, 
    average mapping quality (MAPQ), and read lengths from a BAM file.
    """
    bamfile = pysam.AlignmentFile(bam_path, "rb")
    coverage, avg_mapq, avg_read_length = [], [], []

    for chrom, start, end in tqdm(zip(bins_df['chrom'], bins_df['start'], bins_df['end']), 
                                  total=len(bins_df), desc=f"Processing {os.path.basename(bam_path)}"):
        try:
            reads = list(bamfile.fetch(contig=chrom, start=start, end=end))
            coverage.append(len(reads))
            
            if reads:
                avg_mapq.append(np.mean([r.mapping_quality for r in reads]))
                lengths = [r.query_length for r in reads if r.query_length]
                avg_read_length.append(np.mean(lengths) if lengths else 0)
            else:
                avg_mapq.append(0)
                avg_read_length.append(0)
                
        except ValueError:
            # Triggered if the chromosome does not exist in the BAM index
            coverage.append(0)
            avg_mapq.append(0)
            avg_read_length.append(0)

    bamfile.close()
    return coverage, avg_mapq, avg_read_length

## Execution: Batch Processing

In [ ]:
"""
Executes the extraction functions across all defined disease states and control sets.
"""
bigwig_files = {
    0: "../data/inputs/bigwig/base_ENCFF859SEG.bigWig",
    1: "../data/inputs/bigwig/MCI_ENCFF283XLM.bigWig",
    2: "../data/inputs/bigwig/CI_ENCFF268IZG.bigWig",
    'na': "../data/inputs/bigwig/unseen_ENCFF055LUU.bigWig",
}

bam_files = {
    0: "../data/inputs/bam/base_ENCFF834UNW.bam",
    1: "../data/inputs/bam/MCI_ENCFF424CLY.bam",
    2: "../data/inputs/bam/CI_ENCFF451VHQ.bam",
    'na': "../data/inputs/bam/unseen_ENCFF768ZXA.bam",
}

label_strings = {0: 'Baseline', 1: 'Moderate Cognitive Impairment', 2: 'Cognitive Impairment', 'na': 'Unseen'}
df_dict = {}

for label, file_path in bigwig_files.items():
    dataset_name = label_strings.get(label)
    print(f"\nInitiating extraction for: {dataset_name}")
    
    # Extract structural and signal data
    df = process_bigWig(file_path)
    
    # Extract alignment metrics
    bam_path = bam_files.get(label)
    coverage, mapq, read_len = get_bam_coverage_per_bin(bam_path, df)
    
    # Integrate features
    df['control'] = coverage
    df['avg_mapq'] = mapq
    df['avg_read_length'] = read_len
    
    df_dict[label] = df
    print(f"Extraction complete for {dataset_name} | Shape: {df.shape}")

# Serialize the raw extracted data dictionaries
joblib.dump(df_dict, "../data/saves/Ext_DataFrame_Dict_Base.joblib")
print("\nData serialized successfully to ../data/saves/Ext_DataFrame_Dict_Base.joblib")